# 🎥 Video2LoRA: Zero-Visual-Token Inference Tutorial

Welcome to the official tutorial for **Video2LoRA**! This notebook demonstrates how to run zero-visual-token video inference. By using a perceiver hypernetwork, Video2LoRA reads video features layer-by-layer and predicts custom Low-Rank Adaptation (LoRA) weights. Once injected, the Vision-Language Model (VLM) can answer queries about the video **without requiring visual tokens** in its context window.

### Step 0: Environment Setup & Package Installation

This cell automatically detects whether you are running in **Google Colab** or a **local workspace** and configures the environment accordingly.

In [ ]:
# Detect Google Colab environment
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import os

if IN_COLAB:
    print("Running in Google Colab. Setting up repositories...")
    if not os.path.exists("/content/Video2LoRA"):
        os.system("git clone -b demo https://github.com/video2lora/Video2LoRA.git /content/Video2LoRA")
    if not os.path.exists("/content/video2lora.github.io"):
        os.system("git clone https://github.com/video2lora/video2lora.github.io.git /content/video2lora.github.io")
    os.chdir("/content/Video2LoRA")

# Link website media repository if available
if os.path.lexists("media"):
    if not os.path.isdir("media") or os.path.islink("media"):
        try:
            if os.path.islink("media"):
                os.unlink("media")
            else:
                import shutil
                shutil.rmtree("media")
        except Exception:
            os.system("rm -rf media")

website_media_path = None
possible_paths = [
    "/content/video2lora.github.io/media",
    "/content/Video2LoRA/video2lora.github.io/media",
    "video2lora.github.io/media",
    "../video2lora.github.io/media",
    "../../video2lora.github.io/media",
]
for p in possible_paths:
    abs_p = os.path.abspath(p)
    if os.path.exists(abs_p) and os.path.isdir(abs_p):
        website_media_path = abs_p
        break

if website_media_path:
    try:
        os.symlink(website_media_path, "media")
    except Exception:
        os.system(f"ln -s {website_media_path} media")

print(f"Workspace directory: {os.getcwd()}")

# Install dependencies
!pip install torch torchvision torchaudio
!pip install transformers accelerate peft huggingface_hub decord av opencv-python matplotlib einops jaxtyping num2words "torchao>=0.16.0"

### Step 1: Import Utilities & Patch Environment

In [ ]:
import sys
import os
from pathlib import Path

# Add repository root and src directory to sys.path to enable local imports
repo_root = str(Path(os.getcwd()).resolve())
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
src_dir = str(Path(repo_root) / "src")
if os.path.exists(src_dir) and src_dir not in sys.path:
    sys.path.insert(0, src_dir)

import torch
from scripts.video2lora.train_smolvlm_stage1 import build_stage1_model
from scripts.video2lora.train_smolvlm_online import prepare_smolvlm_inputs

# Import utilities from the demo package
from demo.utils import (
    Video2LoRAConfig,
    patch_num2words,
    download_qualitative_videos,
    show_video_frames,
    run_internalization,
    display_comparison,
    apply_lora_to_layers
)

# Refresh Hugging Face dependency cache for num2words
patch_num2words()

### Step 2: Define and Download Qualitative Examples

In [ ]:
examples = [
    {
        "id": "man-smoking-pipe",
        "video_path": "media/benchmarks/carebench/v_00014063_0.mp4",
        "dataset": "CaReBench: Caption",
        "prompt": "Describe the video in as much useful visual detail as possible. Include the main activity, visible people or objects, scene context, appearance, and any important visual details that help explain what is happening.",
        "target_text": "This video depicts a scene of a man lighting a pipe with a lighter. The man in the video is smoking a pipe held in his mouth, supported by his left hand, while his right hand grips the lighter. His right forearm features a large black tattoo. He has short, thick hair that is a deep brown color and is dressed in a loose-fitting black tank top. He is seated next to a window, which has a wooden frame and blue curtains, with a brick wall behind him and a wooden door on the right. The door has a square pattern, adorned with silver hinges and a doorknob. In the video, he first ignites the lighter with his right hand and then brings the flame to the pipe, holding it in that position for several seconds. Throughout this time, his left hand remains steady on the pipe, and his gaze is fixed intently on it, ensuring that the pipe is fully lit before setting the lighter down. He then continues to hold the pipe with his left hand and begins to smoke. The video is shot from the front, clearly illustrating how relaxed he is while smoking at home."
    },
    {
        "id": "child-watering-plants",
        "video_path": "media/benchmarks/carebench/v_00016555_0.mp4",
        "dataset": "CaReBench: Events",
        "prompt": "Describe the key visible events in chronological order. Include all important actions and changes you can observe, with enough detail to distinguish each event clearly.",
        "target_text": "Little boy watering plants outdoors; Using watering can to pour water into flower pot; Shifting camera angle from side view to rear view; Tapping edge of flower pot a few times; Setting down watering can"
    },
    {
        "id": "posture-alignment",
        "video_path": "media/benchmarks/plm/f522598789220c70_122_155.mp4",
        "dataset": "PLM-SGQA",
        "prompt": "Does this look like the same posture she's holding?",
        "target_text": "Yes, it appears you're mirroring the same posture. Your alignment, knee bend, and spine position match the demonstration, indicating proper form and engagement of the targeted muscle groups for optimal effectiveness and safety."
    },
    {
        "id": "dog-tugging",
        "video_path": "media/benchmarks/plm/b5bdb7f254cb1727_369_400.mp4",
        "dataset": "PLM-SGQA",
        "prompt": "Is he trying to tug?",
        "target_text": "Yes, your dog is likely inviting a tug-of-war game. Holding the toy in his mouth and possibly looking at you or wagging his tail indicates he's ready to engage in a playful tug."
    },
    {
        "id": "rainy-day",
        "video_path": "media/benchmarks/vidcapbench/132065802449.mp4",
        "dataset": "VidCapBench",
        "prompt": "What is the weather like in the scene? Answer only the question, in one sentence.",
        "target_text": "Rainy day."
    },
    {
        "id": "tarsier-creature",
        "video_path": "media/benchmarks/vidcapbench/Tarsier_20.mp4",
        "dataset": "VidCapBench",
        "prompt": "Which parts of the creature are highlighted in the video? Answer only the question, in one sentence.",
        "target_text": "A close-up of its face, eyes, and hair."
    }
]

download_qualitative_videos(examples)

### Step 3: Visualize Video Frames

In [ ]:
# Visualize keyframes for the first qualitative example
show_video_frames(examples[0]["video_path"], num_frames=4)

### Step 4: Download Video2LoRA Checkpoint

In [ ]:
from huggingface_hub import hf_hub_download

print("Downloading pre-trained 2.2B Video2LoRA Checkpoint...")
checkpoint_dir = Path("checkpoints/Video2LoRA-SmolVLM-ckpts")
checkpoint_dir.mkdir(parents=True, exist_ok=True)

ckpt_path = hf_hub_download(
    repo_id="MananSuri27/Video2LoRA-SmolVLM-ckpts",
    filename="video2lora-smolvlm2-2.2b-best-ce.pt",
    local_dir=str(checkpoint_dir)
)
print(f"Checkpoint successfully downloaded to: {ckpt_path}")

### Step 5: Initialize SmolVLM2 & Video2LoRA Models

In [ ]:
# Set configuration arguments matching the 2.2B SmolVLM2 model preset
config = Video2LoRAConfig(
    smolvlm_name_or_path="HuggingFaceTB/SmolVLM2-2.2B-Instruct",
    train_manifest="",
    val_manifest="",
    output_dir="",
    lora_r=16,
    lora_dropout=0.0,
    target_modules=["down_proj"],
    latent_size=512,
    dropout_rate=0.0,
    n_latent_queries=8,
    num_blocks=9,
    num_self_attn_per_block=0,
    video_fps=None,
    max_frames=12,
    video_size_longest_edge=384,
    video_load_backend="auto",
    internalization_prompt="Internalize this video for later captioning.",
    kl_weight=0.0,
    generation_max_new_tokens=128
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Loading SmolVLM2 model...")
model, raw_model, processor, tokenizer = build_stage1_model(config, device=device)

print("Loading hypernetwork weights...")
state_dict = torch.load(ckpt_path, map_location="cpu", weights_only=False)
model.load_state_dict(state_dict)

model.to(device).eval()
raw_model.eval()
print("Model load complete.")

### Step 6: Test Video Internalization

In [ ]:
# Run internalization for Example 2
print(f"Running internalization for Example 2: {examples[1]['id']}...")
sample_loras = run_internalization(examples[1], model, raw_model, processor, config, device)
print("Internalization test successful.")

### Step 7: Base Model Inference (With Visual Tokens)

In [ ]:
import gc

# Reset model state
model.reset()
torch.cuda.empty_cache()
gc.collect()

base_predictions = []
print("Running Base Model Inference (with visual tokens) for all examples...")

with torch.inference_mode():
    for idx, example in enumerate(examples):
        print(f"Processing Example {idx+1}/{len(examples)}: {example['id']}...")
        
        base_messages = [[
            {
                "role": "user",
                "content": [
                    {"type": "video", "path": example["video_path"]},
                    {"type": "text", "text": example["prompt"]}
                ]
            }
        ]]

        base_inputs = prepare_smolvlm_inputs(
            processor,
            base_messages,
            device,
            video_fps=config.video_fps,
            max_frames=config.max_frames,
            video_size_longest_edge=config.video_size_longest_edge,
            video_load_backend=config.video_load_backend
        )

        base_generated_ids = raw_model.generate(
            **base_inputs,
            max_new_tokens=config.generation_max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

        base_pred = tokenizer.decode(
            base_generated_ids[0][base_inputs["input_ids"].shape[1]:],
            skip_special_tokens=True
        ).strip()
        
        base_predictions.append(base_pred)
        
        del base_inputs, base_generated_ids
        torch.cuda.empty_cache()
        gc.collect()

print("\nBase Model inferences complete.")

### Step 8: Video2LoRA Internalization & Inference (Zero Visual Tokens)

In [ ]:
video2lora_predictions = []
print("Running Video2LoRA Inference (zero visual tokens) for all examples...")

with torch.inference_mode():
    for idx, example in enumerate(examples):
        print(f"Processing Example {idx+1}/{len(examples)}: {example['id']}...")
        
        # Generate custom LoRA adapter
        generated_loras = run_internalization(example, model, raw_model, processor, config, device)
        
        # Inject dynamic weights
        model.patch_lora_forward()
        apply_lora_to_layers(
            model.base_model,
            model.hypernet.layer_indices,
            generated_loras,
            torch.ones(1, dtype=torch.int32, device=device),
            position_ids=None
        )
        
        # Query VLM using text only (zero visual tokens)
        prompt_ids = tokenizer.apply_chat_template(
            [{
                "role": "user",
                "content": [{"type": "text", "text": example["prompt"]}]
            }],
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt"
        ).to(device)

        if hasattr(prompt_ids, "keys"):
            input_ids = prompt_ids["input_ids"]
            attention_mask = prompt_ids.get("attention_mask", torch.ones_like(input_ids))
        else:
            input_ids = prompt_ids
            attention_mask = torch.ones_like(input_ids)

        generated_ids = model.base_model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=config.generation_max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

        video2lora_pred = tokenizer.decode(
            generated_ids[0][input_ids.shape[1]:],
            skip_special_tokens=True
        ).strip()
        
        video2lora_predictions.append(video2lora_pred)
        
        del generated_loras, prompt_ids, input_ids, attention_mask, generated_ids
        model.reset()
        torch.cuda.empty_cache()
        gc.collect()

model.reset()
torch.cuda.empty_cache()
gc.collect()
print("\nVideo2LoRA inferences complete.")

### Step 9: Qualitative Comparison Dashboard

In [ ]:
print("=== Summary of Qualitative Responses ===")
for idx, example in enumerate(examples):
    print(f"\n--- Example {idx+1}: {example['id']} ({example['dataset']}) ---")
    print(f"Question: {example['prompt']}")
    print(f"Ground Truth: {example['target_text']}")
    print(f"Base Model  : {base_predictions[idx]}")
    print(f"Video2LoRA  : {video2lora_predictions[idx]}")

print("\nRendering HTML Dashboards...")
for idx, example in enumerate(examples):
    display_comparison(
        video_path=example["video_path"],
        question_prompt=example["prompt"],
        ground_truth=example["target_text"],
        base_model_output=base_predictions[idx],
        video2lora_output=video2lora_predictions[idx],
        dataset_name=example["dataset"]
    )